# **Modelo de Recomendação INERI:**

Este notebook implementa um sistema de recomendação utilizando a técnica de filtragem colaborativa baseada em fatoração de matriz, por meio do algoritmo SVD (Singular Value Decomposition), disponibilizado pela biblioteca Surprise. A filtragem colaborativa é um método que recomenda itens a partir de padrões de comportamento observados em diferentes usuários, partindo do princípio de que usuários que demonstraram preferências semelhantes tendem a manter padrões próximos em suas escolhas futuras.

No contexto deste projeto, cada conta é tratada como um usuário, enquanto cada ativo disponível é considerado um item. As avaliações (ratings) utilizadas pelo modelo são obtidas a partir de uma normalização percentual da alocação financeira de cada ativo dentro da carteira do cliente, permitindo transformar dados financeiros em um formato compatível com técnicas de recomendação.

Embora o conceito inicial de filtragem colaborativa possa ser dividido entre abordagens baseadas em usuário (user-based) ou em item (item-based), o algoritmo SVD pertence à categoria model-based, que trabalha de forma indireta. Em vez de comparar diretamente usuários entre si ou itens entre si, o SVD fatora a matriz de interações entre usuários e itens em representações vetoriais de dimensões latentes, permitindo prever a nota que um usuário daria a um item com base em padrões implícitos da base de dados.

Dessa forma, este projeto utiliza uma abordagem model-based, que considera simultaneamente padrões de comportamento de usuários e itens ao realizar suas previsões de notas e recomendações.

**(Como decidido na AAD, mativemos a database da forma como ela foi enviada)**

---

## **Tópicos**

A seguir, serão apresentadas as seções de **1.0 a 3.0**, que descrevem as etapas para a construção do modelo:

* **1.0 — Identificação de clientes desenquadrados:**
  Nesta etapa, classificamos aproximadamente 80 tipos de ativos distintos e calculamos a proporção de cada um em relação ao investimento total de cada cliente. Em seguida, comparamos essas proporções com os percentuais ideais estabelecidos, identificando assim as carteiras desenquadradas.

* **2.0 — Recomendação de ativos:**
  Desenvolvemos um modelo híbrido, utilizando o algoritmo **SVD** da biblioteca **Surprise**, para recomendar 5 ativos diferentes a cada cliente com carteira desenquadrada. As sugestões são baseadas no comportamento de investimento de outros clientes com perfis semelhantes.

* **3.0 — Análise de resultados:**
  Por fim, avaliamos o desempenho do modelo por meio de uma análise gráfica dos dados e das métricas **RMSE** e **MAE**, validando a precisão e a eficácia das recomendações geradas.


---
# 1.0 Definição de perfis desenquadrados

Esta seção tem como objetivo listar os perfis quem apresentam diferenaça entre perfil de investidor e investimentos reais.

In [41]:
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/My Drive/modelog1"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'Algoritmo de Enquadramento.ipynb'   ModeloGrupo1.ipynb
 ativos_classificados.csv	     PosicaoIntelli.csv
 ativos.csv			     resultado_enquadramento.csv
 carteiras.csv			     resultado_enquadramento_Nome.csv
'CSV importantes'		     Sistema_Recomendação.ipynb


In [42]:
!pip install numpy==1.23.5 #instalação da versão do numpy que não dá conflito com o Surprise
!pip install scikit-surprise
import pandas as pd
import os
import numpy as np
from surprise import Dataset, Reader, SVD
import pickle

### **Importação dos datasets**

In [43]:
carteiras = pd.read_csv("/content/drive/My Drive/modelog1/carteiras.csv")
ativos = pd.read_csv("/content/drive/My Drive/modelog1/ativos.csv")
carteiras.columns = ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'Tipo de Ativo', 'Perfil da carteira']
ativos.columns = ['nome_ativo', 'tipo']
ativos_classificados = pd.read_csv("/content/drive/My Drive/modelog1/ativos_classificados.csv")
carteiras.columns = ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'Tipo de Ativo', 'Perfil da carteira']
ativos_classificados.columns = ['Tipo de Ativo', 'Nivel de Risco']

print(ativos.head())
print(carteiras.head())

    nome_ativo      tipo
0     CRA CDIE  Arrojado
1     CRA IPCA  Arrojado
2  RRRP13 IPCA  Arrojado
3  EEELA1 IPCA  Arrojado
4  TRRG12 IPCA  Arrojado
   Conta    Financeiro  Quantidade   Nome Ativo               Tipo de Ativo  \
0     27  25081.618073        25.0     CRA CDIE  TITULOS RF PRIVADOS BRASIL   
1     27  47046.248881        49.0     CRA IPCA  TITULOS RF PRIVADOS BRASIL   
2     27  36331.476685        35.0  RRRP13 IPCA           DEBENTURES BRASIL   
3     27  18995.623640        20.0  EEELA1 IPCA           DEBENTURES BRASIL   
4     27  39630.685146        39.0  TRRG12 IPCA           DEBENTURES BRASIL   

  Perfil da carteira  
0           Moderado  
1           Moderado  
2           Moderado  
3           Moderado  
4           Moderado  


### **Mapeamento de perfil baseado no Nome do Ativo**

In [44]:
#AQUIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

# Função para mapear a classificação da base ativos.csv
def mapear_classificacao(tipo):
    if tipo == 'Arrojado':
        return 'Renda Variável'
    elif tipo == 'Sofisticado':
        return 'Outro'
    elif tipo == 'Conservador':
        return 'Renda Fixa'
    else:
        return 'Outro'  # Para qualquer outro caso, como segurança

# Fazer o merge entre carteiras e ativos e mapear os tipos
df = carteiras.merge(ativos, left_on='Nome Ativo', right_on='nome_ativo', how='left')
df['tipo_mapeado'] = df['tipo'].apply(mapear_classificacao)

# Função para calcular proporções por carteira
def calcular_proporcoes(df):
    # Agrupar por carteira (Conta) e tipo mapeado, somando os valores financeiros
    agrupado = df.groupby(['Conta', 'Perfil da carteira', 'tipo_mapeado'])['Financeiro'].sum().unstack(fill_value=0)

    # Calcular o total por carteira
    agrupado['total'] = agrupado.sum(axis=1)

    # Calcular proporções
    tipos_ativos = ['Renda Fixa', 'Renda Variável', 'Outro']
    for tipo in tipos_ativos:
        if tipo not in agrupado.columns:
            agrupado[tipo] = 0
        agrupado[f'prop_{tipo}'] = agrupado[tipo] / agrupado['total']

    return agrupado.reset_index()

# Função para verificar enquadramento
def verificar_enquadramento(proporcoes):
    resultados = []

    for _, row in proporcoes.iterrows():
        perfil = row['Perfil da carteira']
        prop_rf = row['prop_Renda Fixa'] if 'prop_Renda Fixa' in row else 0
        prop_rv = row['prop_Renda Variável'] if 'prop_Renda Variável' in row else 0
        prop_outro = row['prop_Outro'] if 'prop_Outro' in row else 0

        enquadrado = False

        if perfil == 'Moderado':
            # 60% Renda Fixa, 40% demais
            enquadrado = (prop_rf >= 0.60) and (prop_rv + prop_outro <= 0.40)

        resultados.append({
            'conta': row['Conta'],
            'perfil': perfil,
            'prop_Renda Fixa': prop_rf,
            'prop_Renda Variável': prop_rv,
            'prop_Outro': prop_outro,
            'enquadrado': 'Sim' if enquadrado else 'Não'
        })

    return pd.DataFrame(resultados)

# Função principal
def main():
    # Calcular proporções
    proporcoes = calcular_proporcoes(df)

    # Verificar enquadramento
    resultado = verificar_enquadramento(proporcoes)

    # Exibir ou salvar resultado
    print(resultado)
    resultado.to_csv('resultado_enquadramento_Nome.csv', index=False)

# Executar
if __name__ == "__main__":
    main()

        conta       perfil  prop_Renda Fixa  prop_Renda Variável  prop_Outro  \
0          27     Moderado          0.00000              1.00000         0.0   
1          44  Sofisticado          0.00000              1.00000         0.0   
2         103     Moderado          0.00000              1.00000         0.0   
3         285  Conservador          0.00000              1.00000         0.0   
4         299  Conservador          0.00000              1.00000         0.0   
...       ...          ...              ...                  ...         ...   
6097  6150189  Conservador          0.00000              1.00000         0.0   
6098  6167502  Conservador          0.40078              0.59922         0.0   
6099  6178808  Sofisticado          0.07664              0.92336         0.0   
6100  6184909  Conservador          1.00000              0.00000         0.0   
6101  6209173     Moderado          1.00000              0.00000         0.0   

     enquadrado  
0           Não  
1  

###**Mapeamento de perfil baseado no Tipo do Ativo**

In [45]:
# Função para mapear o Nivel de Risco para categorias finais
def mapear_classificacao(nivel_risco):
    if nivel_risco == 'Renda Fixa':
        return 'Renda Fixa'
    elif nivel_risco == 'Renda Variável':
        return 'Renda Variável'
    elif nivel_risco == 'Outros':
        return 'Outro'
    else:
        return 'Outro'  # Para segurança

# Fazer o merge entre carteiras e ativos_classificados
df = carteiras.merge(ativos_classificados, on='Tipo de Ativo', how='left')
df['categoria'] = df['Nivel de Risco'].apply(mapear_classificacao)

# Função para calcular proporções por carteira
def calcular_proporcoes(df):
    agrupado = df.groupby(['Conta', 'Perfil da carteira', 'categoria'])['Financeiro'].sum().unstack(fill_value=0)
    agrupado['total'] = agrupado.sum(axis=1)

    tipos_ativos = ['Renda Fixa', 'Renda Variável', 'Outro']
    for tipo in tipos_ativos:
        if tipo not in agrupado.columns:
            agrupado[tipo] = 0
        agrupado[f'prop_{tipo}'] = agrupado[tipo] / agrupado['total']

    return agrupado.reset_index()

# Função para verificar enquadramento e calcular nível de desenquadramento
def verificar_enquadramento(proporcoes):
    resultados = []

    for _, row in proporcoes.iterrows():
        perfil = row['Perfil da carteira']
        prop_rf = row.get('prop_Renda Fixa', 0)
        prop_rv = row.get('prop_Renda Variável', 0)
        prop_outro = row.get('prop_Outro', 0)

        # Define a composição ideal por perfil
        if perfil == 'Conservador':
            ideal_rf, ideal_rv, ideal_outro = 0.9, 0.1, 0.0
        elif perfil == 'Moderado':
            ideal_rf, ideal_rv, ideal_outro = 0.6, 0.4, 0.0
        elif perfil in ['Sofisticado', 'Arrojado']:
            ideal_rf, ideal_rv, ideal_outro = 0.3, 0.7, 0.0
        else:
            ideal_rf, ideal_rv, ideal_outro = 0.0, 0.0, 0.0  # caso sem perfil definido

        # Calcula a distância absoluta para cada proporção
        dist_rf = abs(prop_rf - ideal_rf)
        dist_rv = abs(prop_rv - ideal_rv)
        dist_outro = abs(prop_outro - ideal_outro)

        # Nível de desenquadramento = média das distâncias
        nivel_desenquadramento = (dist_rf + dist_rv + dist_outro) / 3

        # Verifica se está enquadrado conforme a regra anterior
        enquadrado = False
        if perfil == 'Conservador':
            enquadrado = (prop_rf >= 0.90) and (prop_rv + prop_outro <= 0.10)
        elif perfil == 'Moderado':
            enquadrado = (prop_rf >= 0.60 and prop_rf < 0.90) and (prop_rv + prop_outro <= 0.40)
        elif perfil in ['Sofisticado', 'Arrojado']:
            enquadrado = (prop_rv >= 0.70) and (prop_rf + prop_outro <= 0.30)

        resultados.append({
            'conta': row['Conta'],
            'perfil': perfil,
            'prop_Renda Fixa': round(prop_rf, 4),
            'prop_Renda Variável': round(prop_rv, 4),
            'prop_Outro': round(prop_outro, 4),
            'enquadrado': 'Sim' if enquadrado else 'Não',
            'nivel_desenquadramento': round(nivel_desenquadramento, 4)
        })

    return pd.DataFrame(resultados)

# Função principal
def main():
    proporcoes = calcular_proporcoes(df)
    resultado = verificar_enquadramento(proporcoes)

    print(resultado)
    resultado.to_csv('resultado_enquadramento.csv', index=False)
    resultado.to_parquet('resultado_enquadramento.parquet', index=False)

# Executar
if __name__ == "__main__":
    main()


        conta       perfil  prop_Renda Fixa  prop_Renda Variável  prop_Outro  \
0          27     Moderado              1.0                  0.0         0.0   
1          44  Sofisticado              1.0                  0.0         0.0   
2         103     Moderado              1.0                  0.0         0.0   
3         285  Conservador              1.0                  0.0         0.0   
4         299  Conservador              1.0                  0.0         0.0   
...       ...          ...              ...                  ...         ...   
6097  6150189  Conservador              0.0                  1.0         0.0   
6098  6167502  Conservador              1.0                  0.0         0.0   
6099  6178808  Sofisticado              1.0                  0.0         0.0   
6100  6184909  Conservador              1.0                  0.0         0.0   
6101  6209173     Moderado              1.0                  0.0         0.0   

     enquadrado  nivel_desenquadramento

###**Comparar Classificação com base nas duas colunas**

A análise da proporção com base no Tipo de Ativo apresentou resultados mais realistas, isso pode ser explicado pela maior dificuldade de classificar cerca de 8102 ativos individualmente, aumentando a margem para erro e afetando a classificação.

In [46]:
enquadramentoTipo = pd.read_csv("resultado_enquadramento.csv")

print(enquadramentoTipo['enquadrado'].value_counts())

enquadramentoNome = pd.read_csv("resultado_enquadramento_Nome.csv")

print(enquadramentoNome['enquadrado'].value_counts())

enquadrado
Não    4096
Sim    2006
Name: count, dtype: int64
enquadrado
Não    5745
Sim     357
Name: count, dtype: int64


---
# 2.0 Criação do Modelo de Recomendação

Esta seção do notebook realiza tratará da criação do modelo de recomendação, optamos por utilizar a algoritmo de desenquadramento baseado no Tipo de Ativo, por mostrar maior precisão.




## 1️⃣ Carregando os dados
 Dados das carteiras, classificação dos ativos e informações de enquadramento de clientes.

In [47]:
df_enquadramento = pd.read_csv("/content/drive/My Drive/modelog1/resultado_enquadramento.csv")
df_carteiras = pd.read_csv("/content/drive/My Drive/modelog1/carteiras.csv")
df_classificacao = pd.read_csv("/content/drive/My Drive/modelog1/ativos_classificados.csv")

# 2️⃣ Renomeando colunas e ajustando dados
 Ajustamos os nomes das colunas para facilitar o merge e removemos espaços desnecessários.


In [48]:
df_carteiras.columns = ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'tipo_ativo', 'Perfil da carteira']
df_classificacao.columns = ['tipo_ativo', 'Nivel de Risco']
df_classificacao['tipo_ativo'] = df_classificacao['tipo_ativo'].str.strip()

# 3️⃣ Mesclando informações de risco na carteira
 Juntamos o dataframe de carteiras com o de classificação de ativos para trazer o nível de risco.


In [49]:
df_carteiras = df_carteiras.merge(df_classificacao, on='tipo_ativo', how='left')

# 4️⃣ Verificando resultado do merge
 Validamos as colunas resultantes e os valores nulos.


In [50]:
print(f"Colunas após merge: {df_carteiras.columns.tolist()}")
print(f"Valores nulos em 'Nivel de Risco': {df_carteiras['Nivel de Risco'].isna().sum()}")

Colunas após merge: ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'tipo_ativo', 'Perfil da carteira', 'Nivel de Risco']
Valores nulos em 'Nivel de Risco': 3902


# 5️⃣ Tratando valores nulos
 Substituímos riscos não identificados por 'Desconhecido'.

In [51]:
df_carteiras['Nivel de Risco'] = df_carteiras['Nivel de Risco'].fillna('Desconhecido')

# 6️⃣ Separando contas enquadradas e desenquadradas
 Criamos duas listas: contas enquadradas e contas que precisam de ajustes.


In [52]:
contas_enquadradas = df_enquadramento[df_enquadramento['enquadrado'] == 'Sim']['conta']
contas_desenquadradas = df_enquadramento[df_enquadramento['enquadrado'] == 'Não']['conta']

# 7️⃣ Criando rating proporcional para carteiras enquadradas
 Calculamos uma nota normalizada para o financeiro de cada ativo na carteira.


In [53]:
df_treino = df_carteiras[df_carteiras['Conta'].isin(contas_enquadradas)].copy()
df_treino['rating'] = df_treino.groupby('Conta')['Financeiro'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-10)
)

# 8️⃣ Treinamento do modelo de recomendação com Surprise (SVD)
 Montamos o dataset, preparamos o trainset e treinamos o modelo.


In [54]:
import pandas as pd
from surprise import Dataset, Reader, KNNBasic

# Carregar o CSV de proporções
df_enquadramento = pd.read_csv('resultado_enquadramento.csv')

# Transformar as proporções em um formato "long" para Collaborative Filtering
df_props_cf = df_enquadramento.melt(id_vars=['conta'],
                                    value_vars=['prop_Renda Fixa', 'prop_Renda Variável', 'prop_Outro'],
                                    var_name='Nome Ativo', value_name='rating')

# Filtrar apenas onde rating > 0 (pra evitar o ZeroDivisionError)
df_props_cf = df_props_cf[df_props_cf['rating'] > 0]

# Juntar com o dataset de treino original (que já tem Conta, Nome Ativo, rating)
df_treino_filtrado = df_treino[df_treino['rating'] > 0]

# Concatenar os dois
df_treino_expandido = pd.concat([
    df_treino_filtrado[['Conta', 'Nome Ativo', 'rating']].rename(columns={'Conta': 'conta'}),
    df_props_cf
], ignore_index=True)

# Definir o reader para a escala esperada (considerando que vi valor até 1.01)
reader = Reader(rating_scale=(0, 1.2))

# Carregar os dados no formato Surprise
data = Dataset.load_from_df(df_treino_expandido[['conta', 'Nome Ativo', 'rating']], reader)

# Construir o trainset completo
trainset = data.build_full_trainset()

# Definir opções de similaridade (item-item CF ou user-user CF — vou manter item-item como antes)
sim_options = {
    'name': 'pearson_baseline',
    'user_based': False,  # False = item-item CF
    'min_support': 1
}

# Instanciar e treinar o modelo
model = KNNBasic(sim_options=sim_options)
model.fit(trainset)


Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


In [64]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pyarrow as pa
import pyarrow.parquet as pq

# --- Início da Lógica do Código Base ---

# Carregar o CSV de enquadramento
df_enquadramento = pd.read_csv('resultado_enquadramento.csv')

# Transformar as proporções em formato long para CF
df_props_cf = df_enquadramento.melt(id_vars=['conta'],
                                    value_vars=['prop_Renda Fixa', 'prop_Renda Variável', 'prop_Outro'],
                                    var_name='Nome Ativo', value_name='rating')

# Filtrar ratings maiores que zero
df_props_cf = df_props_cf[df_props_cf['rating'] > 0]

# Supondo que 'df_treino' e 'df_carteiras' já estejam carregados no seu ambiente.
# Filtrar o dataset de treino com ratings válidos
df_treino_filtrado = df_treino[df_treino['rating'] > 0]

# Concatenar os dois datasets
df_treino_expandido = pd.concat([
    df_treino_filtrado[['Conta', 'Nome Ativo', 'rating']].rename(columns={'Conta': 'conta'}),
    df_props_cf
], ignore_index=True)

# Pivotar para criar matriz Conta x Nome Ativo, preenchendo NaN com 0
df_pivot = df_treino_expandido.pivot_table(index='conta', columns='Nome Ativo', values='rating', fill_value=0)

# Padronizar os dados para o K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pivot)

# Definir e rodar o K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
df_pivot['cluster'] = kmeans.fit_predict(X_scaled)

# Adicionar a informação de cluster de volta no dataset original
df_treino_expandido = df_treino_expandido.merge(df_pivot[['cluster']], left_on='conta', right_index=True)

# Definir o reader para a escala de rating
reader = Reader(rating_scale=(0, 1.2))

# Criar dicionário de nível de risco e um dataframe de ativos únicos para filtragem
ativos_unicos_df = df_carteiras[['Nome Ativo', 'tipo_ativo', 'Nivel de Risco']].drop_duplicates().reset_index(drop=True)
dict_nivel_risco = dict(zip(ativos_unicos_df['Nome Ativo'], ativos_unicos_df['Nivel de Risco']))

# --- Lógica de Recomendação Modificada e Integrada ---

# Função de rebalanceamento atualizada para retornar perfil e prioridade
def obter_perfil_e_prioridade(conta):
    """Retorna o perfil do cliente e o tipo de ativo prioritário para rebalanceamento."""
    perfil = df_enquadramento[df_enquadramento['conta'] == conta]['perfil'].iloc[0]

    if perfil == 'Conservador':
        return perfil, 'Renda Fixa'
    elif perfil == 'Moderado':
        return perfil, 'Renda Fixa' # Exemplo, pode ser 'Outros' dependendo da regra
    else: # Arrojado / Agressivo
        return perfil, 'Renda Variável'

# Lista para armazenar os registros finais no formato do Parquet
registros_parquet = []

# Rodar o SVD e a nova lógica de recomendação para cada cluster
for cluster_id in sorted(df_treino_expandido['cluster'].unique()):
    print(f"\nTreinando SVD para o Cluster {cluster_id}...")

    df_cluster = df_treino_expandido[df_treino_expandido['cluster'] == cluster_id]

    if len(df_cluster) < 5:
        print(f"Cluster {cluster_id} com poucos dados, ignorado.")
        continue

    data_cluster = Dataset.load_from_df(df_cluster[['conta', 'Nome Ativo', 'rating']], reader)
    trainset_cluster = data_cluster.build_full_trainset()

    model_svd = SVD(random_state=42)
    model_svd.fit(trainset_cluster)

    print(f"Gerando recomendações para as contas do Cluster {cluster_id}...")
    contas_unicas_cluster = df_cluster['conta'].unique()

    # Itera sobre cada conta do cluster para gerar recomendações personalizadas
    for conta in contas_unicas_cluster:
        try:
            # 1. Obter perfil e ativos que o cliente já possui
            perfil_cliente, tipo_prioritario = obter_perfil_e_prioridade(conta)
            ativos_ja_tem = df_carteiras[df_carteiras['Conta'] == conta]['Nome Ativo'].unique()

            # 2. Definir tipos de ativos aceitáveis para o perfil
            if perfil_cliente == 'Conservador':
                tipos_aceitos = ['Renda Fixa']
            elif perfil_cliente == 'Moderado':
                tipos_aceitos = ['Renda Fixa', 'Outros']
            else:  # Arrojado ou Agressivo
                tipos_aceitos = ['Renda Fixa', 'Renda Variável', 'Outros']

            # 3. Filtrar ativos que o cliente não tem e que são adequados ao seu perfil
            mascara = (~ativos_unicos_df['Nome Ativo'].isin(ativos_ja_tem)) & \
                      (ativos_unicos_df['Nivel de Risco'].isin(tipos_aceitos + ['Desconhecido']))

            ativos_validos = ativos_unicos_df[mascara]['Nome Ativo'].values

            # 4. Fazer predições e aplicar bônus de rebalanceamento
            predicoes = []
            for ativo in ativos_validos:
                prediction = model_svd.predict(conta, ativo).est
                nivel_risco = dict_nivel_risco.get(ativo, 'Desconhecido')
                bonus = 0.2 if nivel_risco == tipo_prioritario else 0
                score_final = prediction + bonus
                predicoes.append((ativo, score_final))

            # 5. Obter as 5 melhores recomendações
            top_predicoes = sorted(predicoes, key=lambda x: x[1], reverse=True)[:5]

            # 6. Preparar e armazenar os registros para o Parquet
            if not top_predicoes:
                continue

            # Obter dados de enquadramento da conta uma única vez
            info_enquadramento = df_enquadramento[df_enquadramento['conta'] == conta].iloc[0]

            for ativo, score in top_predicoes:
                registros_parquet.append({
                    'conta': int(conta),
                    'prop_fixa': float(info_enquadramento['prop_Renda Fixa']),
                    'prop_variavel': float(info_enquadramento['prop_Renda Variável']),
                    'prop_outros': float(info_enquadramento['prop_Outro']),
                    'nivel_desenquadramento': float(info_enquadramento['nivel_desenquadramento']),
                    'nome_ativo': ativo,
                    'score': round(score, 3),
                    'recomendacao_ajuste': f"subir_{tipo_prioritario.lower().replace(' ', '_')}"
                })
        except Exception as e:
            print(f"Erro ao processar conta {conta}: {e}")
            continue

# --- Finalização e Salvamento ---

# Criar DataFrame a partir dos registros coletados
df_resultado = pd.DataFrame(registros_parquet)

# Salvar os resultados em Parquet e CSV
if not df_resultado.empty:
    df_resultado.to_parquet('recomendacoes_final.parquet', index=False, engine='pyarrow')
    df_resultado.to_csv('recomendacoes_final.csv', index=False)
    print("\nArquivos recomendacoes_final.parquet e .csv gerados com sucesso.")
else:
    print("\nNenhuma recomendação foi gerada para salvar.")


Treinando SVD para o Cluster 0...
Gerando recomendações para as contas do Cluster 0...

Treinando SVD para o Cluster 1...
Gerando recomendações para as contas do Cluster 1...

Treinando SVD para o Cluster 2...
Gerando recomendações para as contas do Cluster 2...

Arquivos recomendacoes_final.parquet e .csv gerados com sucesso.


In [67]:
import pandas as pd
df_finalrec = pd.read_parquet('recomendacoes_final.parquet', engine='pyarrow')
df_finalrec.head(10)

,conta,prop_fixa,prop_variavel,prop_outros,nivel_desenquadramento,nome_ativo,score,recomendacao_ajuste
0,285,1.0,0.0,0.0,0.0667,LCI PRE,1.006,subir_renda_fixa
1,285,1.0,0.0,0.0,0.0667,CDCA CDIE,0.800,subir_renda_fixa
2,285,1.0,0.0,0.0,0.0667,GASP19 IPCA,0.773,subir_renda_fixa
3,285,1.0,0.0,0.0,0.0667,CSMGB9 IPCA,0.768,subir_renda_fixa
4,285,1.0,0.0,0.0,0.0667,NTNF PRE,0.768,subir_renda_fixa
5,299,1.0,0.0,0.0,0.0667,LCI PRE,0.921,subir_renda_fixa
6,299,1.0,0.0,0.0,0.0667,CDCA CDIE,0.870,subir_renda_fixa
7,299,1.0,0.0,0.0,0.0667,NTNF PRE,0.829,subir_renda_fixa
8,299,1.0,0.0,0.0,0.0667,MRSAC2 IPCA,0.771,subir_renda_fixa
9,299,1.0,0.0,0.0,0.0667,PMEX28,0.763,subir_renda_fixa


# 📦 Salvando o modelo treinado em arquivo pickle

In [68]:
with open('modelo_recomendacao_final.pkl', 'wb') as arquivo_modelo:
    pickle.dump(model, arquivo_modelo)
print("Modelo salvo com sucesso!")

Modelo salvo com sucesso!
